# STFT, Spectrogram and the Time-Frequency Trade-off

This notebook investigates the Short-Time Fourier Transform (STFT) and the spectrogram, and examines the fundamental trade-off between time resolution and frequency resolution.

## Objective

The objectives of this notebook are to:

- Understand why a single FFT cannot tell *when* a frequency component occurs.
- Compute and interpret spectrograms of non-stationary signals.
- Observe how the window length controls time resolution and frequency resolution.
- Separate the effect of window length from the effect of overlap.
- Examine how window type affects dynamic range in a spectrogram.
- Apply the STFT to a frequency-hopping signal and relate window length to dwell time.

## Theory

### Limitation of the FFT

The DFT of an $N$-sample signal is

$$
X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-j 2\pi k n / N}
$$

Every sample of $x[n]$ contributes to every frequency bin. The result describes **which** frequencies are present over the whole observation, but the timing information is spread across the phases of all bins and is not directly visible in $|X[k]|$.

### Short-Time Fourier Transform

The STFT slides a window $w[n]$ of length $L$ along the signal and computes an FFT of each windowed segment:

$$
X[m, k] = \sum_{n=0}^{L-1} x[n + mH]\, w[n]\, e^{-j 2\pi k n / L}
$$

- $m$ : segment (frame) index
- $H$ : hop size, the shift between consecutive segments
- Overlap $= L - H$ samples

The spectrogram is the squared magnitude:

$$
S[m, k] = |X[m, k]|^2
$$

It is usually displayed in dB as an image with time on one axis and frequency on the other. In SDR tools, the same idea is shown as a **waterfall**.

### Time-Frequency Trade-off

For a window of $L$ samples at sampling rate $f_s$:

$$
\Delta t \approx \frac{L}{f_s},
\qquad
\Delta f \approx \frac{f_s}{L}
$$

$$
\Delta t \cdot \Delta f \approx 1
$$

A long window gives fine frequency resolution but blurs events in time. A short window localizes events in time but widens every spectral line. The product cannot be made arbitrarily small; this is the discrete form of the uncertainty principle (Gabor limit).

The window shape changes the constants (main-lobe width, side-lobe level) but not the basic trade-off.

### Overlap

Overlap only changes how densely the time axis is sampled. It does not change $\Delta t$ or $\Delta f$, because each segment still has the same length $L$.

## Setup

Common imports, sampling parameters, and a helper function for plotting spectrograms.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# Sampling frequency
fs = 1000  # Hz

# Reproducible noise
rng = np.random.default_rng(0)


def plot_spectrogram(ax, f, t, Sxx, title, dynamic_range_db=80):
    """Plot a spectrogram in dB with a fixed dynamic range."""
    Sxx_db = 10 * np.log10(Sxx + 1e-20)
    vmax = np.max(Sxx_db)

    mesh = ax.pcolormesh(
        t,
        f,
        Sxx_db,
        shading="auto",
        vmin=vmax - dynamic_range_db,
        vmax=vmax
    )

    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Frequency [Hz]")
    ax.set_title(title)

    return mesh

## Experiment 1: What the FFT Cannot Show

Create two signals that contain the same two frequencies (100 Hz and 250 Hz):

- **Signal A (sequential):** 100 Hz for the first second, then 250 Hz for the second second.
- **Signal B (simultaneous):** 100 Hz and 250 Hz present together for the full two seconds.

Compare their waveforms and their FFT magnitude spectra.

In [ ]:
duration = 2.0  # s
t = np.arange(0, duration, 1 / fs)
N = len(t)

f1 = 100  # Hz
f2 = 250  # Hz

# Signal A: tones one after another
x_seq = np.where(
    t < duration / 2,
    np.sin(2 * np.pi * f1 * t),
    np.sin(2 * np.pi * f2 * t)
)

# Signal B: tones at the same time (scaled to similar total power)
x_sim = (
    np.sin(2 * np.pi * f1 * t)
    + np.sin(2 * np.pi * f2 * t)
) / np.sqrt(2)

print("Number of samples:", N)
print(f"Power of signal A: {np.mean(x_seq**2):.3f}")
print(f"Power of signal B: {np.mean(x_sim**2):.3f}")

In [ ]:
# FFT magnitude spectra
freqs = np.fft.rfftfreq(N, 1 / fs)

X_seq = np.abs(np.fft.rfft(x_seq)) / N
X_sim = np.abs(np.fft.rfft(x_sim)) / N

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

# Time-domain views (zoomed around the transition at t = 1 s)
axes[0, 0].plot(t, x_seq)
axes[0, 0].set_xlim(0.95, 1.05)
axes[0, 0].set_title("Signal A (sequential) - time domain")
axes[0, 0].set_xlabel("Time [s]")
axes[0, 0].grid()

axes[0, 1].plot(t, x_sim)
axes[0, 1].set_xlim(0.95, 1.05)
axes[0, 1].set_title("Signal B (simultaneous) - time domain")
axes[0, 1].set_xlabel("Time [s]")
axes[0, 1].grid()

# Frequency-domain views
axes[1, 0].plot(freqs, X_seq)
axes[1, 0].set_title("Signal A - FFT magnitude")
axes[1, 0].set_xlabel("Frequency [Hz]")
axes[1, 0].set_xlim(0, 400)
axes[1, 0].grid()

axes[1, 1].plot(freqs, X_sim)
axes[1, 1].set_title("Signal B - FFT magnitude")
axes[1, 1].set_xlabel("Frequency [Hz]")
axes[1, 1].set_xlim(0, 400)
axes[1, 1].grid()

plt.tight_layout()
plt.show()

### Observation

- 

## Experiment 2: First Spectrograms

Compute spectrograms of Signal A and Signal B from Experiment 1, and of a linear chirp sweeping from 50 Hz to 400 Hz.

Use a Hann window of 128 samples with 75% overlap for all three.

In [ ]:
# Linear chirp
x_chirp = signal.chirp(
    t,
    f0=50,
    t1=duration,
    f1=400,
    method="linear"
)

nperseg = 128
noverlap = 96

signals = {
    "Signal A (sequential)": x_seq,
    "Signal B (simultaneous)": x_sim,
    "Linear chirp 50 -> 400 Hz": x_chirp,
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

for ax, (name, x) in zip(axes, signals.items()):
    f_s, t_s, Sxx = signal.spectrogram(
        x,
        fs=fs,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        scaling="spectrum"
    )
    plot_spectrogram(ax, f_s, t_s, Sxx, name)

plt.tight_layout()
plt.show()

print(f"Window length: {nperseg} samples = {nperseg / fs * 1000:.0f} ms")
print(f"Bin spacing:   {fs / nperseg:.2f} Hz")

### Observation

- 

## Experiment 3: The Resolution Table

Before changing the window in a spectrogram, calculate the nominal time and frequency resolution for several window lengths.

In [ ]:
window_lengths = [16, 32, 64, 128, 256, 512, 1024]

print(f"{'L [samples]':>12} {'dt [ms]':>10} {'df [Hz]':>10} {'dt*df':>8}")
print("-" * 44)

for L in window_lengths:
    dt = L / fs
    df = fs / L
    print(f"{L:>12} {dt * 1000:>10.1f} {df:>10.2f} {dt * df:>8.2f}")

### Observation

- 

## Experiment 4: Window Length and the Time-Frequency Trade-off

Build a test signal that requires **both** good time resolution and good frequency resolution:

- Two continuous tones that are close in frequency: 200 Hz and 215 Hz.
- Two short clicks (impulses) that are close in time: at 1.00 s and 1.06 s.
- A small amount of white noise.

Compute the spectrogram with three window lengths: 32, 128, and 512 samples.

In [ ]:
# Two close tones
f_a = 200  # Hz
f_b = 215  # Hz

x_test = (
    np.sin(2 * np.pi * f_a * t)
    + np.sin(2 * np.pi * f_b * t)
)

# Two close clicks
click_times = [1.00, 1.06]  # s
click_amplitude = 40

for tc in click_times:
    x_test[int(tc * fs)] += click_amplitude

# Background noise
x_test += 0.05 * rng.standard_normal(N)

print(f"Tone spacing:  {f_b - f_a} Hz")
print(f"Click spacing: {(click_times[1] - click_times[0]) * 1000:.0f} ms")

In [ ]:
test_lengths = [32, 128, 512]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

stft_results = {}

for ax, L in zip(axes, test_lengths):
    f_s, t_s, Sxx = signal.spectrogram(
        x_test,
        fs=fs,
        window="hann",
        nperseg=L,
        noverlap=L - max(L // 16, 1),  # small hop for a dense time axis
        scaling="spectrum"
    )

    stft_results[L] = (f_s, t_s, Sxx)

    plot_spectrogram(
        ax,
        f_s,
        t_s,
        Sxx,
        f"L = {L} ({L / fs * 1000:.0f} ms, {fs / L:.1f} Hz)",
        dynamic_range_db=60
    )

    ax.set_xlim(0.8, 1.3)

plt.tight_layout()
plt.show()

### Observation

- 

## Experiment 5: Slicing the Spectrogram

Images can be misleading, so take two quantitative slices from the spectrograms of Experiment 4:

- **Frequency slice:** the spectrum in a frame far away from the clicks (around t = 0.5 s). Can the two tones be separated?
- **Time slice:** the total energy above 300 Hz versus time. The tones contribute little there, so the clicks dominate. Can the two clicks be separated?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for L in test_lengths:
    f_s, t_s, Sxx = stft_results[L]

    # Frequency slice: frame closest to t = 0.5 s
    i_frame = np.argmin(np.abs(t_s - 0.5))
    spectrum_db = 10 * np.log10(Sxx[:, i_frame] + 1e-20)

    axes[0].plot(
        f_s,
        spectrum_db - np.max(spectrum_db),
        label=f"L = {L}"
    )

    # Time slice: energy above 300 Hz
    band = f_s > 300
    band_energy = np.sum(Sxx[band, :], axis=0)

    axes[1].plot(
        t_s,
        band_energy / np.max(band_energy),
        label=f"L = {L}"
    )

axes[0].axvline(f_a, color="k", linestyle=":", linewidth=1)
axes[0].axvline(f_b, color="k", linestyle=":", linewidth=1)
axes[0].set_xlim(150, 265)
axes[0].set_ylim(-60, 5)
axes[0].set_xlabel("Frequency [Hz]")
axes[0].set_ylabel("Relative power [dB]")
axes[0].set_title("Frequency slice at t = 0.5 s")
axes[0].grid()
axes[0].legend()

for tc in click_times:
    axes[1].axvline(tc, color="k", linestyle=":", linewidth=1)
axes[1].set_xlim(0.8, 1.3)
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Normalized energy above 300 Hz")
axes[1].set_title("Time slice: high-frequency energy")
axes[1].grid()
axes[1].legend()

plt.tight_layout()
plt.show()

### Observation

- 

## Experiment 6: Overlap Is Not Resolution

Keep the window length fixed at 128 samples and change only the overlap: 0%, 50%, and 94%.

Use the chirp from Experiment 2, zoomed into a short time interval, and count the number of frames produced in each case.

In [ ]:
L = 128
overlaps = [0, 64, 120]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

for ax, ov in zip(axes, overlaps):
    f_s, t_s, Sxx = signal.spectrogram(
        x_chirp,
        fs=fs,
        window="hann",
        nperseg=L,
        noverlap=ov,
        scaling="spectrum"
    )

    hop = L - ov

    plot_spectrogram(
        ax,
        f_s,
        t_s,
        Sxx,
        f"overlap = {ov} ({ov / L * 100:.0f}%), hop = {hop}",
        dynamic_range_db=60
    )

    ax.set_xlim(0.5, 1.0)
    ax.set_ylim(100, 300)

    print(
        f"overlap = {ov:>3} samples | hop = {hop:>3} samples "
        f"({hop / fs * 1000:>5.1f} ms) | frames = {len(t_s)}"
    )

plt.tight_layout()
plt.show()

### Observation

- 

## Experiment 7: Window Type and Dynamic Range

The previous notebook showed spectral leakage in a single FFT. The same effect exists in every STFT frame.

Create a strong tone at 200 Hz and a weak tone 60 dB below it at 260 Hz. Compare spectrograms using rectangular, Hann, and Blackman-Harris windows with the same window length.

In [ ]:
strong_amp = 1.0
weak_amp = 1e-3  # -60 dB relative to the strong tone

x_dr = (
    strong_amp * np.sin(2 * np.pi * 200 * t)
    + weak_amp * np.sin(2 * np.pi * 260 * t)
)

x_dr += 1e-5 * rng.standard_normal(N)

window_types = ["boxcar", "hann", "blackmanharris"]
L = 256

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

for ax, win in zip(axes, window_types):
    f_s, t_s, Sxx = signal.spectrogram(
        x_dr,
        fs=fs,
        window=win,
        nperseg=L,
        noverlap=L // 2,
        scaling="spectrum"
    )

    plot_spectrogram(
        ax,
        f_s,
        t_s,
        Sxx,
        f"window = {win}",
        dynamic_range_db=100
    )

    ax.set_ylim(100, 350)

plt.tight_layout()
plt.show()

In [ ]:
# Average spectrum over all frames for each window type
plt.figure(figsize=(10, 5))

for win in window_types:
    f_s, t_s, Sxx = signal.spectrogram(
        x_dr,
        fs=fs,
        window=win,
        nperseg=L,
        noverlap=L // 2,
        scaling="spectrum"
    )

    avg_db = 10 * np.log10(np.mean(Sxx, axis=1) + 1e-20)

    plt.plot(
        f_s,
        avg_db - np.max(avg_db),
        label=win
    )

plt.axvline(260, color="k", linestyle=":", linewidth=1, label="weak tone")
plt.xlim(150, 350)
plt.ylim(-140, 5)
plt.xlabel("Frequency [Hz]")
plt.ylabel("Relative power [dB]")
plt.title("Average STFT spectrum: strong 200 Hz tone, weak 260 Hz tone")
plt.grid()
plt.legend()
plt.show()

### Observation

- 

## Experiment 8: Frequency Hopping and Dwell Time

A frequency-hopping signal jumps between channels, staying on each channel for a fixed **dwell time**. Spectrum monitoring and EW receivers often detect such signals in a waterfall display.

Generate a hopping signal with:

- 8 channels between 100 Hz and 450 Hz
- a dwell time of 40 ms
- additive noise

Compute spectrograms with window lengths of 16, 64, and 256 samples and compare the window duration with the dwell time.

In [ ]:
dwell_time = 0.040  # s
dwell_samples = int(dwell_time * fs)

channels = np.linspace(100, 450, 8)

n_hops = N // dwell_samples
hop_sequence = rng.choice(channels, size=n_hops)

# Build the instantaneous frequency and integrate it to get phase
# (continuous phase, no jumps at hop boundaries)
inst_freq = np.repeat(hop_sequence, dwell_samples)
inst_freq = np.pad(inst_freq, (0, N - len(inst_freq)), mode="edge")

phase = 2 * np.pi * np.cumsum(inst_freq) / fs
x_fh = np.cos(phase)

# Noise (SNR around 10 dB per sample)
x_fh += 0.22 * rng.standard_normal(N)

print(f"Dwell time:  {dwell_time * 1000:.0f} ms ({dwell_samples} samples)")
print(f"Number of hops: {n_hops}")
print(f"Channels [Hz]: {np.round(channels, 1)}")
print(f"First 10 hops [Hz]: {np.round(hop_sequence[:10], 1)}")

In [ ]:
fh_lengths = [16, 64, 256]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

for ax, L in zip(axes, fh_lengths):
    f_s, t_s, Sxx = signal.spectrogram(
        x_fh,
        fs=fs,
        window="hann",
        nperseg=L,
        noverlap=L - max(L // 8, 1),
        scaling="spectrum"
    )

    plot_spectrogram(
        ax,
        f_s,
        t_s,
        Sxx,
        f"L = {L} ({L / fs * 1000:.0f} ms) vs dwell {dwell_time * 1000:.0f} ms",
        dynamic_range_db=40
    )

    ax.set_xlim(0, 0.6)

# True hop pattern for reference (first 0.6 s)
t_hops = np.arange(n_hops) * dwell_time
for ax in axes:
    ax.step(
        t_hops,
        hop_sequence,
        where="post",
        color="w",
        linewidth=0.8,
        linestyle="--"
    )

plt.tight_layout()
plt.show()

### Observation

- 

## Key Takeaways

-

## Next Step

The next notebook will investigate correlation, the matched filter, and energy detection, focusing on how a known waveform can be detected in noise and how detection thresholds relate to false alarm and detection probability.